In [13]:
import argparse
import torch
import os
import json
from tqdm import tqdm
import shortuuid

# 需要定义llava模块的位置，即指定PATH变量
import sys
llava_module_dir = "/root/userfolder/MIL/VL-MIL"
sys.path.insert(0, llava_module_dir)

from llava.constants import IMAGE_TOKEN_INDEX
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path 

from feature_utils import generate_hidden_states

In [14]:
nfi_desc_path = "/root/commonfile/NFI/jsons/screenings"
model_path = "/root/userfolder/data-ckpts/VL-MIL/checkpoints/v2/llava/merged/llava-onevision-qwen2-7b-si-ft-3epoch-NFD-freeze_backbone_valid_1gpu"
image_folder = "/root/commonfile/InfantVQA"
device='cuda'

In [15]:
nfi_list= []
for split in ['train', 'valid', 'test']:
    json_path = os.path.join(nfi_desc_path, f"nfi_{split}.json")
    assert os.path.exists(json_path)
    with open(json_path, 'r') as f:
        nfi_list.extend(json.load(f))
len(nfi_list)

9260

In [16]:
# 匹配某个文件在文件夹中的序号
def get_image_idx(file:str) -> int:
    ndots = file.count(".")
    if ndots == 2:
        return int(file.split(".")[-2])
    elif ndots == 1:
        substr = file.split(".")[-2]
        s = len(substr) - 1
        while("0" <= substr[s] <= "9" and s >= 0):
            s -= 1
        return int(substr[s + 1:])
    else:
        return None

In [17]:
nfi_list[0]

{'id': 1,
 'num_image': 20,
 'patient': 'NEG/ai_bingeying_1800076_20180111',
 'image': ['nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.1.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.2.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.3.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.4.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.5.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.6.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.7.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.8.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd227794a.9.jpg',
  'nfi/images/NEG/ai_bingeying_1800076_20180111/c1285436-a7e8-44ab-ac7f-506dd

In [18]:
patient = nfi_list[0]
def form_conversation(patient):
    pid = patient['id']
    name = patient['patient'].split("/")[-1]
    img_num = patient['num_image']
    images = patient['image']
    
    patient_sample = []

    for img in images:
        img_idx = get_image_idx(img)
        sample = {
            "id": pid + img_idx,
            "image": img,
            "patient": name,
            "description": None,
            "conversations": [
                {
                    "from": "human",
                    "value": "<image>\nPlease tell me the possible abnormality in the newborn fundus image in Chinese."
                },
                {
                    "from": "gpt",
                    "value": "<think>分析过程。</think> <answer>未知。</answer>."
                }
            ]
        }
    
        patient_sample.append(sample)
    return patient_sample 

nfi_desc_dataset = [form_conversation(p) for p in nfi_list]
print(f"NFI dataset has totally {len(nfi_desc_dataset)} patients.")
len(form_conversation(nfi_list[1]))

NFI dataset has totally 9260 patients.


18

加载模型

In [19]:
model_name = get_model_name_from_path(model_path)
tokenizer, model, image_processor, context_len = load_pretrained_model(model_path, None, model_name, device_map=device)

{'device_map': 'cuda'}
Loaded LLaVA model: /root/userfolder/data-ckpts/VL-MIL/checkpoints/v2/llava/merged/llava-onevision-qwen2-7b-si-ft-3epoch-NFD-freeze_backbone_valid_1gpu


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading vision tower: /root/userfolder/model_weights/siglip-so400m-patch14-384


/root/userfolder/anaconda3/envs/llava/lib/python3.10/site-packages/torch/nn/modules/module.py:2025: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(f'for {key}: copying from a non-meta parameter in the checkpoint to a meta '
/root/userfolder/anaconda3/envs/llava/lib/python3.10/site-packages/torch/nn/modules/module.py:2025: UserWarning: for vision_model.embeddings.patch_embedding.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(f'for {key}: copying from a non-met

Model Class: LlavaQwenForCausalLM


这幅图是所有图像 token 的熵变化（标题有误）

Layer -1 表示 SigLip 输出，Layer 0-27 表示 Qwen 第 0 到 27 层的输出，但是在 hidden_states 里面的序号仍是

$l \leftarrow l+1$

![](./asserts/Entropy_based_analysis.png)

In [ ]:
layers = [3, 11, 23, 27, 28]
for layer in layers:
    depth = str(layer) if layer > 10 else f'0{layer}'
    save_path = f'/root/commonfile/InfantVQA/nfi/features/qwen_layer_{depth}_valid'
    os.makedirs(save_path, exist_ok=True)
siglip_path = f'/root/commonfile/InfantVQA/nfi/features/siglip_valid'
os.makedirs(siglip_path, exist_ok=True)
output_path = f"/root/commonfile/InfantVQA/nfi/features/output_valid"
os.makedirs(output_path, exist_ok=True)

In [ ]:
save_path
assert False

AssertionError: 

按照病人进行多图描述

In [ ]:
from torch.nn.utils.rnn import pad_sequence

pbar = tqdm(nfi_desc_dataset, desc=save_path)
for sample in pbar:
    pbar.set_description(sample[0]['patient'])  # 动态更新描述
    pbar.refresh()  # 立即刷新显示
    output_feature_path = os.path.join(output_path, sample[0]['patient'] + '.pt')
        
    # resume
    if os.path.exists(output_feature_path):
        continue
    siglip_feature_list = []
    qwen_feature_list = {l: [] for l in layers}
    output_feature_list = []
    # A sample contains multiple conversations turns of multiple images
    for line in sample:
        patient = line['patient']
         
        features = generate_hidden_states(line, model, tokenizer, image_processor)
        siglip_feature_list.append(features["siglip_features"])
        for layer in layers:
            qwen_feature_list[layer].append(features['input_hidden_states'][layer].cpu().detach())
        output_feature_list.append(features['output_hidden_states'][-1].cpu().detach())
            
    
    # save siglip feature
    siglip_features = torch.stack(siglip_feature_list, dim=0).squeeze(0)
    torch.save(siglip_features, os.path.join(siglip_path, patient + '.pt'))
    # save qwen feature
    for layer in layers:
        concated_features = torch.stack(qwen_feature_list[layer], dim=0).squeeze(0)
        depth = str(layer) if layer > 10 else f'0{layer}'
        save_path = f'/root/commonfile/InfantVQA/nfi/features/qwen_layer_{depth}_valid'
        torch.save(concated_features, os.path.join(save_path, patient + '.pt'))
    
    # the last layer of output states
    output_features = pad_sequence(
        output_feature_list,
        batch_first = True,
        padding_value = 0
    )
    torch.save(
        output_features,
        os.path.join(output_path, patient + '.pt')
    )

zu_peixiaoying_1602541_20161116: 100%|██████████| 9260/9260 [33:27<00:00,  4.61it/s]                 
